# 5. Stochastic Extension of the Lorenz System

The preceding sections developed the Lorenz system as a purely deterministic dynamical system: given an initial condition $\mathbf{x}(0)$, the trajectory $\mathbf{x}(t)$ is uniquely determined for all future time by the ordinary differential equations

$$
\dot{\mathbf{x}} = \mathbf{f}(\mathbf{x}).
$$

In any physical realization, however, the system is inevitably subject to random perturbations. Electronic components exhibit thermal noise, analog multipliers introduce gain fluctuations, and communication channels corrupt transmitted signals. These disturbances are not merely a practical inconvenience; they fundamentally alter the mathematical character of the system, transforming a deterministic ODE into a *stochastic* differential equation (SDE). Understanding this transformation is essential for assessing whether chaotic synchronization — and, by extension, chaotic communication — remains viable under realistic conditions.

## 5.1 Physical Origins of Noise in Chaotic Circuits

Before introducing the mathematical formalism, it is worth cataloguing the physical mechanisms that inject randomness into a Lorenz circuit implementation. Each mechanism contributes noise with distinct statistical characteristics, but for the purposes of a first analysis they can all be aggregated into a single additive white-noise term.

### 5.1.1 Thermal (Johnson–Nyquist) Noise

Every resistor in the Lorenz circuit is a source of thermal noise. At thermal equilibrium, the random motion of charge carriers produces a fluctuating voltage across the resistor terminals. The power spectral density of this voltage is given by the Nyquist formula,

$$
S_V(f) = 4 k_B T R,
$$

where $k_B$ is Boltzmann's constant, $T$ is the absolute temperature, and $R$ is the resistance. This spectrum is flat (white) up to frequencies on the order of $k_B T / h \approx 6 \times 10^{12}$ Hz at room temperature, far above the bandwidth of any practical Lorenz circuit. Consequently, thermal noise is well modeled as additive white Gaussian noise (AWGN) within the circuit's operating band.

The Cuomo transmitter circuit employs over twenty resistors with values ranging from $1\,\text{k}\Omega$ to $200\,\text{k}\Omega$. Each contributes an independent noise source. When these are referred to the state-variable nodes $(u, v, w)$ through the operational-amplifier network, they produce effective noise voltages at each integrator output. Because the resistor values differ, the noise intensities at the three nodes are in general unequal, though for a simplified analysis one often adopts a single scalar noise intensity $\sigma_n$ as a representative magnitude.

### 5.1.2 Operational-Amplifier Noise

The operational amplifiers (LF353 in the Cuomo circuit) contribute two kinds of noise: an equivalent input voltage noise and an equivalent input current noise. The voltage noise is typically broadband with a spectral density on the order of $10$–$20\;\text{nV}/\sqrt{\text{Hz}}$ for a JFET-input amplifier, while the current noise is smaller but not negligible at high impedance nodes. Additionally, the AD632 analog multipliers used to implement the nonlinear terms $uv$ and $uw$ introduce their own broadband noise floor. Since these active-device noise sources are also spectrally flat within the circuit bandwidth, they reinforce the appropriateness of a white-noise model.

### 5.1.3 Channel Noise

When the Lorenz circuit is used for communication, the drive signal $u(t)$ must travel from the transmitter to the receiver through a physical channel — a wire, a radio link, or an optical fiber. Each of these channels adds noise. In the simplest model, the received signal is

$$
u_{\text{received}}(t) = u(t) + \nu(t),
$$

where $\nu(t)$ is a zero-mean white Gaussian process with spectral density $N_0/2$. This channel noise enters the receiver subsystem as an external perturbation to the drive input, and its effect on synchronization quality is one of the central questions motivating the stochastic analysis.

### 5.1.4 Component Tolerances and Parameter Mismatch

A subtler form of randomness arises from manufacturing tolerances. No two resistors, capacitors, or multipliers are identical. If the transmitter has parameters $(\sigma, r, b)$ and the receiver has slightly different values $(\sigma + \delta\sigma, r + \delta r, b + \delta b)$, perfect synchronization is impossible even in the absence of additive noise. Cuomo et al. investigated this numerically (their Fig. 13) and found that the Lorenz synchronizing system is reasonably robust to modeling errors up to about 10–15%, beyond which the output signal-to-noise ratio degrades significantly. While parameter mismatch is not the same as additive noise, it can be partially absorbed into the stochastic framework by treating the mismatch as a slowly varying random perturbation.

## 5.2 Mathematical Preliminaries: The Wiener Process and Itô Calculus

To place the noisy Lorenz system on rigorous mathematical footing, we require the language of stochastic calculus. The fundamental building block is the *Wiener process*, also known as standard Brownian motion.

### 5.2.1 Definition and Properties of the Wiener Process

A standard Wiener process $W(t)$, with $t \geq 0$, is a continuous-time stochastic process satisfying the following properties:

1. **Initial condition:** $W(0) = 0$ almost surely.

2. **Independent increments:** For any $0 \leq t_1 < t_2 \leq t_3 < t_4$, the increments $W(t_2) - W(t_1)$ and $W(t_4) - W(t_3)$ are independent random variables.

3. **Gaussian increments:** Each increment $W(t+\Delta t) - W(t)$ is normally distributed with mean zero and variance $\Delta t$:
$$
W(t + \Delta t) - W(t) \sim \mathcal{N}(0,\, \Delta t).
$$

4. **Continuous paths:** The sample paths $t \mapsto W(t)$ are continuous with probability one, but they are *nowhere differentiable*. This last property is crucial: one cannot write $dW/dt$ in the ordinary sense, which is precisely why stochastic calculus differs from classical calculus.

The variance of $W(t)$ itself grows linearly with time,

$$
\text{Var}[W(t)] = \mathbb{E}[W(t)^2] = t,
$$

which means the Wiener process diffuses away from the origin at a rate proportional to $\sqrt{t}$. In discrete approximation, over a time step $\Delta t$, the Wiener increment is

$$
\Delta W = W(t + \Delta t) - W(t) = \sqrt{\Delta t}\;\xi, \qquad \xi \sim \mathcal{N}(0, 1),
$$

a fact that will be important when we discuss numerical simulation methods.

### 5.2.2 The Itô Stochastic Differential Equation

A general Itô SDE for a state vector $\mathbf{x}(t) \in \mathbb{R}^n$ takes the form

$$
d\mathbf{x} = \mathbf{f}(\mathbf{x}, t)\,dt + \mathbf{G}(\mathbf{x}, t)\,d\mathbf{W}_t,
$$

where $\mathbf{f}: \mathbb{R}^n \to \mathbb{R}^n$ is the *drift* vector (the deterministic part), $\mathbf{G}: \mathbb{R}^n \to \mathbb{R}^{n \times m}$ is the *diffusion* matrix (governing how noise enters each equation), and $\mathbf{W}_t = (W_1(t), \ldots, W_m(t))^\top$ is an $m$-dimensional vector of independent Wiener processes.

The notation $d\mathbf{x}$ is shorthand for the integral equation

$$
\mathbf{x}(t) = \mathbf{x}(0) + \int_0^t \mathbf{f}(\mathbf{x}(s), s)\,ds + \int_0^t \mathbf{G}(\mathbf{x}(s), s)\,d\mathbf{W}_s,
$$

where the second integral is an *Itô integral*. The Itô integral is defined as a limit of sums in which the integrand is evaluated at the *left endpoint* of each subinterval, in contrast to the Stratonovich convention which uses the midpoint. This choice is not merely a technicality; it determines the form of the chain rule.

### 5.2.3 Itô's Lemma

In ordinary calculus, if $x(t)$ satisfies an ODE and $V(x)$ is a smooth function, the chain rule gives $dV/dt = V'(x)\,\dot{x}$. In stochastic calculus the chain rule acquires an additional second-order correction. For a scalar SDE $dx = f\,dt + g\,dW$ and a twice-differentiable function $V(x, t)$, Itô's lemma states

$$
dV = \left(\frac{\partial V}{\partial t} + f\frac{\partial V}{\partial x} + \frac{1}{2}g^2 \frac{\partial^2 V}{\partial x^2}\right)dt + g\frac{\partial V}{\partial x}\,dW.
$$

The extra term $\tfrac{1}{2}g^2 V''$ — absent from the deterministic chain rule — is a direct consequence of the fact that $(dW)^2 = dt$ to leading order. This lemma is the principal computational tool for analyzing functions of stochastic processes, and we will invoke it when studying how noise modifies Lyapunov functions for the synchronization error.

## 5.3 Stochastic Lorenz Equations

We are now in a position to write down the stochastic extension of the circuit-scaled Lorenz system. The simplest and most commonly adopted model adds independent white noise to each state equation with a uniform intensity $\sigma_n > 0$:

### 5.3.1 Additive Noise Model

$$
\boxed{
\begin{aligned}
du &= \sigma(v - u)\,dt + \sigma_n\,dW_1, \\
dv &= (ru - v - 20uw)\,dt + \sigma_n\,dW_2, \\
dw &= (5uv - bw)\,dt + \sigma_n\,dW_3,
\end{aligned}
}
$$

where $W_1, W_2, W_3$ are three mutually independent standard Wiener processes, and $\sigma_n$ is the noise intensity (in the same voltage units as $u$, $v$, $w$). This is an *additive* noise model because the diffusion coefficient $\sigma_n$ does not depend on the state $\mathbf{x}$. In the language of the general Itô SDE, the drift is the familiar Lorenz vector field $\mathbf{f}(\mathbf{x})$ and the diffusion matrix is $\mathbf{G} = \sigma_n \mathbf{I}_3$, the scalar $\sigma_n$ times the $3 \times 3$ identity.

The additive model is appropriate when the dominant noise sources are exogenous (thermal noise, channel noise) rather than state-dependent. For situations where noise enters multiplicatively — for instance, if the gain of an analog multiplier fluctuates in proportion to its output — one would instead write

$$
du = \sigma(v - u)\,dt + \sigma_n\,u\,dW_1,
$$

which is a *multiplicative* noise model. Multiplicative noise introduces qualitatively different behavior (noise-induced transitions, modified fixed-point locations), but for the Lorenz circuit the additive model is the standard first approximation and the one we adopt throughout.

### 5.3.2 Non-Uniform Noise Intensities

In a more refined model, each state variable may experience a different noise intensity, reflecting the different impedance environments in the circuit:

$$
\begin{aligned}
du &= \sigma(v - u)\,dt + \sigma_{n,1}\,dW_1, \\
dv &= (ru - v - 20uw)\,dt + \sigma_{n,2}\,dW_2, \\
dw &= (5uv - bw)\,dt + \sigma_{n,3}\,dW_3.
\end{aligned}
$$

Here the diffusion matrix becomes $\mathbf{G} = \text{diag}(\sigma_{n,1},\, \sigma_{n,2},\, \sigma_{n,3})$. One can estimate the individual $\sigma_{n,i}$ from the circuit topology by summing the noise contributions of all resistors referred to each integrator node. For the purposes of this theoretical development, however, we continue with the uniform case $\sigma_{n,1} = \sigma_{n,2} = \sigma_{n,3} = \sigma_n$ to keep expressions tractable.

## 5.4 Effects of Noise on Chaotic Dynamics

Adding noise to the Lorenz system has several profound consequences for the nature of the dynamics. These effects are not merely quantitative (trajectories get "fuzzy") but qualitative: the very framework for describing the system must change.

### 5.4.1 Trajectory Perturbation and Attractor Blurring

In the deterministic Lorenz system, the strange attractor is a fractal set of measure zero in $\mathbb{R}^3$: all trajectories eventually settle onto this thin, intricately folded structure. When additive noise is present, trajectories are continually kicked off the attractor by random impulses and then pulled back by the deterministic flow. The result is that the system no longer visits a sharp fractal set but instead occupies a *neighborhood* of the attractor whose width scales with $\sigma_n$.

Quantitatively, the instantaneous deviation $\delta \mathbf{x}$ induced by noise satisfies, to linear order,

$$
d(\delta \mathbf{x}) = \mathbf{J}(\mathbf{x}(t))\,\delta \mathbf{x}\,dt + \sigma_n\,d\mathbf{W}_t,
$$

where $\mathbf{J}(\mathbf{x})$ is the Jacobian of $\mathbf{f}$ evaluated along the (now stochastic) trajectory. Along directions where the Lorenz system is contracting (negative local Lyapunov exponent), perturbations are suppressed; along the expanding direction (positive Lyapunov exponent $\lambda_1 > 0$), perturbations are amplified. The net effect is that the attractor acquires a "thickness" that is largest along the unstable manifold and smallest along the most stable direction.

### 5.4.2 Loss of Deterministic Predictability

The deterministic Lorenz system is already unpredictable in a practical sense: small errors in initial conditions grow exponentially, with a characteristic doubling time of $1/\lambda_1$. Noise worsens this by continually injecting new uncertainty at every instant. Even if the initial condition were known exactly, the future trajectory would still be uncertain because the noise realization is unknown.

This means that in the stochastic setting, one must abandon the notion of predicting individual trajectories and instead work with *ensembles* of trajectories. The relevant objects become probability distributions $p(\mathbf{x}, t)$ rather than deterministic solutions $\mathbf{x}(t)$. The evolution of this probability density is governed by the Fokker–Planck equation, which we develop in Section 5.5.

### 5.4.3 Noise-Induced Effects on Invariant Measures

The deterministic Lorenz attractor supports a natural invariant measure $\mu_0$ — informally, the long-time probability of finding the system in any given region of state space. When noise is added, this measure is replaced by a *stationary probability density* $p_s(\mathbf{x})$ that solves the stationary Fokker–Planck equation. For small $\sigma_n$, this density is concentrated near the deterministic attractor and approximately proportional to $\mu_0$ convolved with a Gaussian kernel of width $\sim \sigma_n$. For larger $\sigma_n$, the density spreads further, and can even "fill in" regions between the two lobes of the attractor that are rarely visited deterministically, qualitatively changing the statistics of the system.

## 5.5 The Fokker–Planck Equation

While the SDE describes the evolution of individual (stochastic) trajectories, the Fokker–Planck equation (FPE) describes the evolution of the probability density $p(\mathbf{x}, t)$ of the state vector. For the stochastic Lorenz system with additive noise, the FPE takes the form

$$
\frac{\partial p}{\partial t} = -\sum_{i=1}^{3} \frac{\partial}{\partial x_i}\bigl[f_i(\mathbf{x})\,p\bigr] + \frac{\sigma_n^2}{2}\sum_{i=1}^{3}\frac{\partial^2 p}{\partial x_i^2},
$$

or in compact vector notation,

$$
\frac{\partial p}{\partial t} = -\nabla \cdot (\mathbf{f}\,p) + \frac{\sigma_n^2}{2}\nabla^2 p.
$$

The first term on the right is the *advection* of probability by the deterministic flow: if there were no noise, $p$ would simply be transported along the Lorenz vector field, and any initial delta function $p(\mathbf{x}, 0) = \delta(\mathbf{x} - \mathbf{x}_0)$ would remain a delta function riding along the trajectory. The second term is *diffusion*: noise spreads probability out in all directions at a rate proportional to $\sigma_n^2/2$.

The Fokker–Planck equation is a *linear* partial differential equation for $p$, even though the underlying SDE is nonlinear in $\mathbf{x}$. This linearity is a significant theoretical advantage: while individual trajectories diverge chaotically, the probability density evolves smoothly and predictably. In principle, one can find the stationary density $p_s(\mathbf{x})$ by solving $\partial p / \partial t = 0$, although for the three-dimensional Lorenz system this generally requires numerical methods.

## 5.6 Noise and Synchronization: Stochastic Stability Analysis

The central question for chaotic communications is whether synchronization between the transmitter and receiver survives in the presence of noise. In the deterministic case (developed in Section 3), the synchronization error $\mathbf{e} = \mathbf{x}_{\text{transmitter}} - \mathbf{x}_{\text{receiver}}$ converges exponentially to zero, as demonstrated by the Lyapunov function

$$
E(\mathbf{e}) = \frac{1}{2}\left(\frac{e_1^2}{\sigma} + e_2^2 + 4e_3^2\right),
$$

whose time derivative was shown to be negative definite. In the stochastic case, perfect synchronization ($\mathbf{e} = 0$) is no longer achievable; instead, the error fluctuates around zero, and the relevant question becomes: how large are these fluctuations?

### 5.6.1 Stochastic Error Dynamics

Consider the case where additive noise enters both the transmitter and the channel. The transmitter evolves according to the stochastic Lorenz equations, and the receiver receives the noisy drive signal $u(t) + \nu(t)$. The error dynamics become

$$
\begin{aligned}
de_1 &= \sigma(e_2 - e_1)\,dt + \sigma_n\,dW_1^{(\text{ch})}, \\
de_2 &= (-e_2 - 20u\,e_3)\,dt + \sigma_n\,dW_2, \\
de_3 &= (5u\,e_2 - b\,e_3)\,dt + \sigma_n\,dW_3,
\end{aligned}
$$

where $dW_1^{(\text{ch})}$ represents channel noise entering through the drive signal, and $dW_2$, $dW_3$ represent noise within the receiver circuit itself. The crucial difference from the deterministic case is that even when $\mathbf{e}$ reaches the neighborhood of zero, the noise terms continually perturb it away.

### 5.6.2 Itô's Lemma Applied to the Lyapunov Function

Applying Itô's lemma to the Lyapunov function $E(\mathbf{e})$, we obtain

$$
dE = \underbrace{\left(\nabla E \cdot \mathbf{f}_e\right)}_{\text{deterministic (stabilizing)}}\,dt + \underbrace{\frac{\sigma_n^2}{2}\,\text{tr}\left(\mathbf{G}^\top \nabla^2 E\;\mathbf{G}\right)}_{\text{noise injection (destabilizing)}}\,dt + \text{(martingale term)}\,d\mathbf{W},
$$

where $\mathbf{f}_e$ is the drift of the error dynamics and $\nabla^2 E$ is the Hessian of $E$. The first $dt$ term is the same expression that was shown to be negative definite in the deterministic analysis — it represents the stabilizing tendency of synchronization. The second $dt$ term is new: it is always *positive* and represents the continuous injection of energy into the error by the noise.

For the Lyapunov function $E = \frac{1}{2}(e_1^2/\sigma + e_2^2 + 4e_3^2)$ with uniform noise $\mathbf{G} = \sigma_n \mathbf{I}$, the Hessian is $\nabla^2 E = \text{diag}(1/\sigma,\, 1,\, 4)$ and the noise-injection term evaluates to

$$
\frac{\sigma_n^2}{2}\left(\frac{1}{\sigma} + 1 + 4\right) = \frac{\sigma_n^2}{2}\left(\frac{1}{\sigma} + 5\right).
$$

With $\sigma = 16$, this gives approximately $\frac{\sigma_n^2}{2}(5.0625) \approx 2.53\,\sigma_n^2$.

### 5.6.3 Steady-State Error Bound

In the deterministic case, the time derivative $\dot{E}$ satisfies

$$
\dot{E} \leq -\gamma\, E,
$$

for some effective decay rate $\gamma > 0$ that depends on the Lorenz parameters. In the stochastic case, taking expectations and using the fact that the Itô integral has zero expectation, we get

$$
\frac{d}{dt}\mathbb{E}[E] \leq -\gamma\,\mathbb{E}[E] + \frac{\sigma_n^2}{2}\left(\frac{1}{\sigma} + 5\right).
$$

This is a first-order linear ODE for $\mathbb{E}[E]$. Its steady-state solution ($d\mathbb{E}[E]/dt = 0$) gives

$$
\boxed{
\mathbb{E}[E]_{\text{ss}} = \frac{\sigma_n^2}{2\gamma}\left(\frac{1}{\sigma} + 5\right).
}
$$

This is a key result. It says that the mean-squared synchronization error is proportional to $\sigma_n^2$ (the noise power) and inversely proportional to $\gamma$ (the synchronization strength). Synchronization is never perfect in the presence of noise, but the error can be made small if the noise is weak relative to the natural stability of the synchronization manifold.

The transient solution, starting from an initial error $\mathbb{E}[E](0) = E_0$, is

$$
\mathbb{E}[E](t) = E_0\,e^{-\gamma t} + \frac{\sigma_n^2}{2\gamma}\left(\frac{1}{\sigma} + 5\right)\left(1 - e^{-\gamma t}\right),
$$

which shows exponential convergence from $E_0$ to the noise floor $\mathbb{E}[E]_{\text{ss}}$.

### 5.6.4 Signal-to-Noise Ratio and Communication Quality

For the signal masking scheme, the recovered message is $\hat{m}(t) = s(t) - u_r(t) = m(t) + e_1(t)$, so the message recovery error is precisely the synchronization error $e_1$. The output signal-to-noise ratio is therefore

$$
\text{SNR}_{\text{out}} = \frac{\mathbb{E}[m^2]}{\mathbb{E}[e_1^2]} \approx \frac{P_m}{\sigma_n^2 / (\sigma\gamma)} = \frac{P_m \sigma \gamma}{\sigma_n^2},
$$

where $P_m = \mathbb{E}[m^2]$ is the message power. This expression makes the trade-offs explicit: higher synchronization strength $\gamma$ and higher $\sigma$ improve message recovery, while stronger noise degrades it quadratically.

Cuomo et al. found experimentally that for their circuit with $\sigma = 16$, $r = 45.6$, $b = 4$, the Lorenz synchronizing system and the extended Kalman filter produced comparable output SNRs over a wide range of input SNRs, with both exhibiting a threshold effect below which message recovery fails entirely. This threshold corresponds roughly to the regime where $\mathbb{E}[e_1^2]$ becomes comparable to the variance of the chaotic signal itself, at which point the receiver can no longer distinguish the synchronization error from the drive signal.

## 5.7 Numerical Methods for Stochastic Differential Equations

Standard deterministic integrators such as the fourth-order Runge–Kutta method (RK4) are not directly applicable to SDEs, because they do not account for the $O(\sqrt{dt})$ scaling of the noise increments. Specialized stochastic integrators are required, and the two most widely used are presented here.

### 5.7.1 The Euler–Maruyama Method

The Euler–Maruyama method is the stochastic analogue of the forward Euler method. For the SDE $d\mathbf{x} = \mathbf{f}(\mathbf{x})\,dt + \mathbf{G}\,d\mathbf{W}_t$, the discrete update is

$$
\mathbf{x}_{n+1} = \mathbf{x}_n + \mathbf{f}(\mathbf{x}_n)\,\Delta t + \mathbf{G}\,\Delta \mathbf{W}_n,
$$

where $\Delta \mathbf{W}_n = \sqrt{\Delta t}\;\boldsymbol{\xi}_n$ and each $\xi_{n,i} \sim \mathcal{N}(0,1)$ independently. This method has *strong order of convergence* $1/2$, meaning the expected error in any individual trajectory scales as $O(\sqrt{\Delta t})$, and *weak order of convergence* $1$, meaning errors in expected values of smooth functions of the solution scale as $O(\Delta t)$.

For the stochastic Lorenz system with $\sigma = 16$, $r = 45.6$, $b = 4$, and $\mathbf{G} = \sigma_n \mathbf{I}$, the explicit Euler–Maruyama update reads

$$
\begin{aligned}
u_{n+1} &= u_n + \sigma(v_n - u_n)\,\Delta t + \sigma_n \sqrt{\Delta t}\;\xi_{n,1}, \\
v_{n+1} &= v_n + (r\,u_n - v_n - 20\,u_n w_n)\,\Delta t + \sigma_n \sqrt{\Delta t}\;\xi_{n,2}, \\
w_{n+1} &= w_n + (5\,u_n v_n - b\,w_n)\,\Delta t + \sigma_n \sqrt{\Delta t}\;\xi_{n,3}.
\end{aligned}
$$

The time step $\Delta t$ must be chosen small enough to resolve both the deterministic dynamics (which for the Lorenz system with these parameter values have a characteristic timescale on the order of the inverse of the largest eigenvalue of the Jacobian, roughly $\sim 1/\sigma = 1/16$) and the noise fluctuations.

### 5.7.2 The Milstein Method

The Milstein method improves on Euler–Maruyama by including a correction term that accounts for the state-dependence of the diffusion coefficient. For a general scalar SDE $dx = f(x)\,dt + g(x)\,dW$, the Milstein update is

$$
x_{n+1} = x_n + f(x_n)\,\Delta t + g(x_n)\,\Delta W_n + \frac{1}{2}g(x_n)\,g'(x_n)\left[(\Delta W_n)^2 - \Delta t\right].
$$

The additional term $\frac{1}{2}g\,g'[(\Delta W)^2 - \Delta t]$ arises from a Taylor expansion of the diffusion term and raises the strong order of convergence to $1$.

For the stochastic Lorenz system with *additive* noise ($g = \sigma_n = \text{const}$, so $g' = 0$), the Milstein correction vanishes identically, and the method reduces to Euler–Maruyama. This is a fortunate simplification: it means that for our additive-noise model, there is no advantage to using Milstein over Euler–Maruyama. The distinction becomes important only if we adopt a multiplicative noise model, where $g$ depends on the state variables.

It is worth emphasizing that this equivalence is *specific to additive noise*. If one were to model, say, state-dependent thermal noise where the noise intensity scales with current (and hence with the state variable), the Milstein method would provide a genuine improvement in accuracy.

### 5.7.3 Higher-Order Stochastic Integrators

Beyond Euler–Maruyama and Milstein, there exist higher-order methods such as the stochastic Runge–Kutta schemes of Rößler and the strong-order 1.5 methods of Kloeden and Platen. These require generating correlated random increments (not just independent Gaussian samples) and are considerably more complex to implement. For the Lorenz system with moderate noise intensities, Euler–Maruyama with a sufficiently small time step (typically $\Delta t \leq 10^{-4}$) provides adequate accuracy, and we recommend it as the default choice for the computational notebook.

### 5.7.4 Convergence and Stability Considerations

A critical distinction in stochastic numerics is between *strong* and *weak* convergence. Strong convergence measures the pathwise error $\mathbb{E}[|\mathbf{x}(t) - \mathbf{x}_N|]$, which tracks how well the numerical solution approximates a specific trajectory for a given noise realization. Weak convergence measures the error in expectations $|\mathbb{E}[\phi(\mathbf{x}(t))] - \mathbb{E}[\phi(\mathbf{x}_N)]|$ for smooth test functions $\phi$, which tracks statistical properties.

For the synchronization analysis, weak convergence is usually sufficient: we care about the *mean-squared* error $\mathbb{E}[E]$ rather than the error on any single realization. This means Euler–Maruyama (weak order 1) is quite adequate.

Stability of the numerical scheme is also a concern. For the Lorenz system, the largest eigenvalue of the Jacobian can reach $|\lambda| \sim \sigma = 16$, imposing a stability limit of roughly $\Delta t < 2/\sigma \approx 0.125$ for explicit methods. In practice, one uses a time step much smaller than this (e.g., $\Delta t = 10^{-3}$ to $10^{-4}$) to also resolve the fine structure of the attractor.

## 5.8 Comparison: Lorenz SCS vs. Extended Kalman Filter

Cuomo et al. draw an illuminating comparison between the Lorenz synchronizing chaotic system (SCS) and the extended Kalman filter (EKF) as competing approaches to state estimation in the presence of noise.

The Lorenz SCS is an *open-loop* nonlinear observer: the receiver equations use the drive signal $u(t)$ directly but do not adjust their gains based on the estimation error. This makes the SCS simple to implement (it is essentially a copy of the transmitter circuit with the drive signal fed in) but limits its ability to optimize noise rejection.

The EKF, by contrast, is a *closed-loop* estimator that linearizes the system about the current state estimate and uses the Riccati equation to compute an optimal gain $K(t)$ at each instant. The EKF minimizes the mean-squared estimation error under the assumption of Gaussian noise, making it theoretically optimal for small perturbations. However, the EKF requires accurate knowledge of the system parameters and the noise statistics, and it can diverge if the initial state estimate is poor — the chaotic sensitivity means that a bad initial guess leads to linearization about a wildly incorrect trajectory.

The SCS has the advantage of *global* convergence: synchronization occurs from any initial condition, as guaranteed by the Lyapunov analysis. The EKF has the advantage of better noise filtering above its convergence threshold. In practice, Cuomo et al. found that the two estimators perform comparably over a wide range of input SNRs, with the SCS being slightly more robust to initial-condition errors and the EKF being slightly better at high SNR.

## 5.9 Summary and Bridge to the Computational Notebook

This section has extended the deterministic Lorenz system to the stochastic domain, motivated by the physical reality of noise in electronic circuits and communication channels. The key theoretical results are:

1. The stochastic Lorenz system is an Itô SDE with additive noise, characterized by the noise intensity $\sigma_n$.

2. Noise blurs the strange attractor, destroys deterministic predictability, and replaces individual trajectories with probability densities governed by the Fokker–Planck equation.

3. Synchronization persists in the presence of noise, but the error converges to a nonzero steady-state mean-squared value $\mathbb{E}[E]_{\text{ss}} \propto \sigma_n^2 / \gamma$.

4. The output SNR for signal masking is $\text{SNR}_{\text{out}} \propto P_m \sigma \gamma / \sigma_n^2$, making explicit the trade-off between message power, synchronization strength, and noise intensity.

5. The Euler–Maruyama method is the appropriate numerical integrator for the additive-noise case, coinciding with the Milstein method since the diffusion coefficient is state-independent.

The computational notebook will implement these theoretical predictions numerically: simulating the stochastic Lorenz system, verifying attractor blurring, measuring synchronization error as a function of $\sigma_n$, and demonstrating message recovery under noisy conditions.